# Segundo modelo: clasificador de huevos con YOLOv8-cls (CDD_eggs_SJ)

Dataset: [CDD_eggs_SJ](https://universe.roboflow.com/sj-ytwh0/cdd_eggs_sj) — 5.305 imágenes originales,
3 clases (`good`, `broken`, `dirty`), licencia **CC BY 4.0** (hay que citarlo en el informe; la cita está al final).

## Lo que hay que saber antes de correr esto

**1. Es un dataset de CLASIFICACIÓN, no de detección.** No trae cajas. Por eso se entrena
`yolov8m-cls` (una etiqueta por imagen), no `yolov8m` detector. En el proyecto encaja como
**segunda etapa**: el detector que ya tienes encuentra el huevo → se recorta → este modelo lo juzga.

**2. La versión publicada tiene fuga entre splits.** Hay frames del mismo video
(`20251002_174109_007`, `_028`, `_034`…) y ráfagas de fotos con segundos de diferencia. Un split aleatorio
pone un frame en train y su vecino en test: el modelo "ya vio" ese huevo y las métricas salen infladas.
Este notebook **agrupa todo lo que es el mismo huevo** (copias aumentadas, frames de video, ráfagas y
casi-duplicados visuales) y reparte **grupos**, no imágenes.

**3. Split 70/15/15, no 80/10/10.** Con ~5.3k originales, los ~530 originales que se pierden de train
cambian poco a un modelo preentrenado, pero un test 50 % más grande estrecha el intervalo de confianza de
la **fuga** (el KPI del proyecto), y un val más grande hace más estable el early stopping. Además, al
repartir grupos (que vienen en bloques) hace falta holgura para que cada clase quede bien estratificada.

**4. La versión v2 viene estirada a 224×224 y con shear ±10° horneado.** Se entrena a 224 (la resolución
nativa de `-cls`). Para la máxima calidad, ver la nota al final sobre hacer *fork* y exportar sin
aumentaciones.

## Qué responde el modelo

El modelo aprende 3 clases (`Roto`, `Sano`, `Sucio`), pero la **decisión** es jerárquica, como en una
planta real: primero *¿está dañado o no?*, y solo si está bueno, *¿limpio o sucio?*

| veredicto | cuándo | en la banda |
|---|---|---|
| ✗ **DAÑADO** | p(Roto) ≥ `TAU_ROTO` | descarte (servo) |
| ⚠ **REVISIÓN** | `TAU_DUDA` ≤ p(Roto) < `TAU_ROTO` | inspección humana |
| ✓ **BUENO (SUCIO)** | p(Roto) < `TAU_DUDA` y p(Sucio) > p(Sano) | lavado → empaque |
| ✓ **BUENO** | p(Roto) < `TAU_DUDA` y p(Sano) ≥ p(Sucio) | empaque |

Todo se decide con **un solo número**, p(Roto), y dos cortes. Se explica en una frase: *"si el modelo
cree que hay entre X % y Y % de probabilidad de que esté roto, lo mira una persona"*. Los dos cortes se
calibran en **val**, nunca en test:
- `TAU_ROTO`: el que minimiza el costo (un roto que se cuela cuesta 12 veces botar uno bueno).
- `TAU_DUDA`: la banda de revisión más ancha que no mande a revisión más del 5 % de los huevos buenos.

La revisión por **foto borrosa** ("no lo ve bien") ya la hace el backend con la compuerta de nitidez.

## Antes de empezar
1. **Entorno de ejecución → Cambiar tipo de entorno → T4 GPU.**
2. En roboflow.com → *Settings → API Keys* copia la **Private API Key** (la de la tabla, *Full access*).
   **No** la *Publishable* que empieza con `rf_`: esa es solo para inferencia en el navegador.
3. En Colab: panel izquierdo → icono de **llave (Secretos)** → **+ Agregar nuevo secreto** →
   nombre `ROBOFLOW_API_KEY`, valor = tu Private API Key → activa **Acceso al notebook**.
   Así la clave nunca queda escrita en el notebook.
4. **Entorno de ejecución → Ejecutar todas.** Tarda ~35–45 min en T4.
   - Al principio pide **permiso para Google Drive**: acéptalo. Se pide al inicio a propósito: si lo
     pidiera al final, tras 40 minutos sin que estés, la sesión podría cerrarse y perderías el modelo.
   - El entrenamiento se guarda **directo en Drive** época a época: si Colab se desconecta, no pierdes lo hecho.
   - La última celda te pide **subir fotos tuyas** de huevos para ver cómo las clasifica.

No pongas tu *Workspace ID* en ningún lado: el dataset pertenece al workspace `sj-ytwh0` (su autor),
y tu clave solo se usa para autorizar la descarga.

In [ ]:
# @title 1. Entorno: GPU + librerías (mismas versiones que tu primer entrenamiento)
import subprocess
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print(gpu or "SIN GPU -> Entorno de ejecucion > Cambiar tipo de entorno > T4 GPU")
assert gpu, "Activa la GPU antes de seguir"

# ultralytics 8.4.157 = la misma version con la que se entreno el detector: el export ONNX sale igual.
%pip install -q ultralytics==8.4.157 roboflow==1.5.0 onnx onnxruntime onnxslim
import ultralytics
ultralytics.checks()

In [ ]:
# @title 2. Configuración (todo lo ajustable está aquí)
# Workspace del AUTOR del dataset (no el tuyo): tu API key solo autoriza la descarga.
WORKSPACE = "sj-ytwh0"
PROYECTO  = "cdd_eggs_sj"
VERSION   = 2                     # la ultima publicada

# Como mapear las 3 clases del dataset a las del proyecto:
#   3clases            broken->Roto, good->Sano, dirty->Sucio   (recomendado)
#   binario            broken->Roto, good+dirty->Sano           (sucio no es dano)
#   binario_sin_sucios broken->Roto, good->Sano, dirty se descarta
MODO_CLASES = "3clases"  # @param ["3clases", "binario", "binario_sin_sucios"]

PROPORCIONES = {"train": 0.70, "val": 0.15, "test": 0.15}

MODELO_BASE = "yolov8m-cls.pt"  # @param ["yolov8n-cls.pt", "yolov8s-cls.pt", "yolov8m-cls.pt", "yolov8l-cls.pt"]
IMGSZ    = 224    # resolucion nativa de -cls y de la v2 del dataset
BATCH    = 64
SEMILLA  = 42

# Entrenamiento largo: minimo de epocas, y parada solo tras PACIENCIA epocas sin una
# mejora SIGNIFICATIVA (>= MEJORA_MINIMA de exactitud en val). Ultralytics no trae
# ninguna de las dos cosas: las implementa un callback en la celda 9.
MIN_EPOCAS    = 150    # @param {type:"integer"}
PACIENCIA     = 30     # @param {type:"integer"}
MEJORA_MINIMA = 0.005  # @param {type:"number"}
EPOCAS_MAX    = 200    # @param {type:"integer"}

# Sobremuestreo en TRAIN de clases minoritarias hasta esta fraccion de la mayoritaria.
# (cls_pw NO sirve en clasificacion en 8.4.157: la perdida es cross_entropy sin pesos.)
SOBREMUESTREO = 0.6

# Una sola copia por original tambien en train. La v2 trae cada huevo de train DUPLICADO
# (flip+shear horneados): con eso cada "epoca" son ~2 pasadas por huevo y un entrenamiento
# de 150+ epocas memoriza el doble de rapido. Las aumentaciones en linea ya cubren flip y shear.
UNA_COPIA_EN_TRAIN = True

# Matriz de costos del proyecto: dejar pasar un roto cuesta 12 veces botar uno bueno.
COSTO_FN, COSTO_FP = 12.0, 1.0

# Carga maxima de REVISION: que fraccion de huevos BUENOS (limpios o sucios) se tolera
# mandar a inspeccion humana. Con esto se calibra la banda de duda en val.
MAX_REVISION_BUENOS = 0.05  # @param {type:"number"}

GUARDAR_EN_DRIVE = True  # @param {type:"boolean"}

from pathlib import Path
from datetime import datetime
NOMBRE_RUN = f"cdd_{MODO_CLASES}_{Path(MODELO_BASE).stem}_{datetime.now():%Y%m%d_%H%M}"
ORIGEN  = Path("/content/cdd_original")
LIMPIO  = Path("/content/cdd_limpio")

# Drive se monta AL INICIO a proposito: si el permiso se pidiera al final, tras 40 min de
# entrenamiento sin nadie frente a la pantalla, la sesion podria cerrarse y perder el modelo.
# Ademas el run se guarda directo en Drive, epoca a epoca: sobrevive a una desconexion.
if GUARDAR_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA_RUNS = Path("/content/drive/MyDrive/eggdetection/runs_cdd")
else:
    CARPETA_RUNS = Path("/content/runs_cdd")
CARPETA_RUNS.mkdir(parents=True, exist_ok=True)
print("run:", NOMBRE_RUN, "\nse guarda en:", CARPETA_RUNS / NOMBRE_RUN)

### 3. Biblioteca de split sin fuga
Esta celda escribe `cdd_split_lib.py`, la lógica que agrupa "el mismo huevo" y reparte grupos.
Es **exactamente** el archivo probado localmente con `test_cdd_split_lib.py` sobre un dataset sintético
con fugas plantadas (copias aumentadas, frames de video, ráfagas, duplicados recomprimidos y volteados).

In [ ]:
%%writefile cdd_split_lib.py
# =============================================================================
#  Split limpio para datasets de CLASIFICACION exportados de Roboflow
#  (probado localmente con colab/test_cdd_split_lib.py antes de ir a Colab)
#
#  El problema que resuelve: un split aleatorio por imagen mete fuga cuando
#  hay (a) copias aumentadas del mismo original, (b) frames del mismo video,
#  (c) rafagas de fotos del mismo huevo o (d) duplicados subidos dos veces.
#  Si un frame cae en train y su vecino en test, el modelo "ya vio" ese huevo
#  y las metricas salen infladas: se ve genial y falla en campo.
#
#  Solucion: agrupar todo lo que es "el mismo huevo" y repartir GRUPOS, no
#  imagenes, estratificando por clase.
#
#  v2 — correccion tras el primer entrenamiento real:
#  El hash de 64 bits no separa "misma foto" de "huevo distinto con el mismo
#  fondo" (medido en 968 fotos reales: duplicados hasta 6 bits, el 5 % de los
#  huevos distintos a <=5 bits). Encadenado con union-find, eso formo UN grupo
#  con el 91.5 % del dataset y dejo val/test con 453 originales en vez de ~1.590.
#  Ahora: hash de 256 bits (duplicados p95=13, distintos p5=29), las uniones por
#  hash tienen TOPE de tamano, y revisar_reparto() se niega a seguir si un split
#  queda desbalanceado.
# =============================================================================
from __future__ import annotations

import random
import re
import shutil
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
from PIL import Image

EXT_IMG = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
CARPETAS_SPLIT = {"train": "train", "valid": "val", "val": "val", "validation": "val", "test": "test"}


# ------------------------------------------------------------------ nombres
def nombre_base(nombre: str) -> str:
    """'20251002_174109_028_jpg.rf.ab12cd.jpg' -> '20251002_174109_028'.

    Roboflow renombra 'foto.jpg' a 'foto_jpg.rf.<hash>.jpg'. Todas las copias
    aumentadas de un mismo original comparten lo que va antes de '.rf.'.
    """
    base = nombre.split(".rf.")[0]
    base = re.sub(r"\.(jpe?g|png|bmp|webp)$", "", base, flags=re.I)
    base = re.sub(r"_(jpe?g|png|bmp|webp)$", "", base, flags=re.I)
    return base


RE_VIDEO = re.compile(r"^(\d{8}_\d{6})_(\d{1,5})$")                          # 20251002_174109_028
RE_FECHA = re.compile(r"^(\d{4})-(\d{2})-(\d{2})-(\d{2})_(\d{2})_(\d{2})(?:_(\d+))?$")  # 2023-04-09-20_19_15_972818
RE_FRAME = re.compile(r"^(.+?)[_\-\s]*frame[_\-\s]*\d+$", re.I)              # clip3_frame_0045


def clave_video(base: str) -> str | None:
    """Frames de un mismo video comparten prefijo: agruparlos es obligatorio."""
    if m := RE_VIDEO.match(base):
        return "video:" + m.group(1)
    if m := RE_FRAME.match(base):
        return "video:" + m.group(1).lower()
    return None


def fecha_de(base: str) -> datetime | None:
    if m := RE_FECHA.match(base):
        y, mo, d, h, mi, s, micro = m.groups()
        return datetime(int(y), int(mo), int(d), int(h), int(mi), int(s),
                        int((micro or "0")[:6].ljust(6, "0")))
    return None


# ------------------------------------------------------------ hash perceptual
def _dhash_de(im: Image.Image, lado: int) -> np.ndarray:
    a = np.asarray(im.resize((lado + 1, lado), Image.BILINEAR), dtype=np.int16)
    bits = np.packbits((a[:, 1:] > a[:, :-1]).flatten())          # lado*lado bits
    return bits.view(np.uint64).copy()                             # lado=16 -> 4 x uint64


def dhash_par(ruta: Path, lado: int = 16) -> tuple[np.ndarray, np.ndarray]:
    """(dhash, dhash del espejo horizontal) con UNA sola decodificacion.

    lado=16 da 256 bits. Con 64 bits (lado=8) los duplicados verdaderos y los
    huevos distintos fotografiados sobre el mismo fondo se solapan: no existe
    un umbral que los separe (medido sobre fotos reales).
    """
    im = Image.open(ruta).convert("L")
    return _dhash_de(im, lado), _dhash_de(im.transpose(Image.FLIP_LEFT_RIGHT), lado)


def calcular_hashes(items: list[dict], lado: int = 16, log=print) -> None:
    log(f"  hash perceptual ({lado * lado} bits) de {len(items)} imagenes...")
    for it in items:
        it["h"], it["hf"] = dhash_par(it["ruta"], lado)


_LUT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


def _popcount(x: np.ndarray) -> np.ndarray:
    """Bits en 1 por fila. x: (..., k) uint64 -> (...,)"""
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(x).sum(axis=-1, dtype=np.int32)
    return _LUT[x.view(np.uint8)].sum(axis=-1, dtype=np.int32)


class UnionFind:
    """Union-find con tamanos (para el tope) y causas por grupo (para el diagnostico)."""

    def __init__(self, n: int) -> None:
        self.p = list(range(n))
        self.tam = [1] * n
        self.causas: list[Counter] = [Counter() for _ in range(n)]

    def find(self, x: int) -> int:
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x

    def tam_de(self, x: int) -> int:
        return self.tam[self.find(x)]

    def union(self, a: int, b: int, causa: str) -> bool:
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.tam[ra] < self.tam[rb]:
            ra, rb = rb, ra
        self.p[rb] = ra
        self.tam[ra] += self.tam[rb]
        self.causas[ra] += self.causas[rb]
        self.causas[ra][causa] += 1
        return True


# ------------------------------------------------------------------ inventario
def inventariar(raiz: Path) -> list[dict]:
    """Lee un export 'folder' de Roboflow: <raiz>/<split>/<clase>/<imagen>."""
    items = []
    for carpeta_split in sorted(p for p in raiz.iterdir() if p.is_dir()):
        split = CARPETAS_SPLIT.get(carpeta_split.name.lower())
        if split is None:
            continue
        for carpeta_clase in sorted(p for p in carpeta_split.iterdir() if p.is_dir()):
            for f in sorted(carpeta_clase.iterdir()):
                if f.suffix.lower() in EXT_IMG:
                    items.append({"ruta": f, "split_orig": split, "clase_orig": carpeta_clase.name,
                                  "base": nombre_base(f.name)})
    return items


def agrupar(items: list[dict], gap_rafaga_s: float = 20.0, duracion_max_rafaga_s: float = 120.0,
            umbral_hash: int = 16, max_grupo_hash: int = 30, usar_hash: bool = True, log=print) -> dict:
    """Asigna 'grupo' a cada item uniendo cuatro fuentes de 'es el mismo huevo'.

    Las tres primeras (nombre del original, video, rafaga) son evidencia fuerte y
    se aplican siempre. El hash visual es evidencia debil: se aplica de la pareja
    mas parecida a la menos, y NUNCA si la union produciria un grupo de mas de
    `max_grupo_hash` imagenes. Los duplicados reales vienen en parejas o grupos
    chicos; un grupo de miles solo puede salir de encadenar huevos distintos.

    Requiere calcular_hashes(items) antes si usar_hash=True.
    """
    n = len(items)
    uf = UnionFind(n)
    razones = Counter()

    def unir(i, j, causa):
        if uf.union(i, j, causa):
            razones[causa] += 1

    # 1) copias aumentadas del mismo original (mismo nombre antes de '.rf.')
    por_base = defaultdict(list)
    for i, it in enumerate(items):
        por_base[it["base"]].append(i)
    for idx in por_base.values():
        for j in idx[1:]:
            unir(idx[0], j, "copia aumentada")

    # 2) frames de un mismo video
    por_video = defaultdict(list)
    for i, it in enumerate(items):
        if k := clave_video(it["base"]):
            por_video[k].append(i)
    for idx in por_video.values():
        for j in idx[1:]:
            unir(idx[0], j, "frame del mismo video")

    # 3) rafagas: fotos con marca de tiempo a menos de gap_rafaga_s entre si.
    #    Con tope de duracion: sin el, alguien que fotografio cada 10 s durante
    #    una hora terminaria con un solo grupo gigante (encadenamiento).
    fechadas = sorted(((fecha_de(it["base"]), i) for i, it in enumerate(items) if fecha_de(it["base"])),
                      key=lambda t: t[0])
    inicio = fechadas[0][0] if fechadas else None
    for (t0, i0), (t1, i1) in zip(fechadas, fechadas[1:]):
        if (t1 - t0).total_seconds() <= gap_rafaga_s and (t1 - inicio).total_seconds() <= duracion_max_rafaga_s:
            unir(i0, i1, f"rafaga (<{gap_rafaga_s:g}s)")
        else:
            inicio = t1

    # 4) casi-duplicados visuales, de la pareja mas parecida a la menos, con tope
    rechazadas = 0
    if usar_hash and n > 1:
        h = np.stack([it["h"] for it in items])
        hf = np.stack([it["hf"] for it in items])
        pares = []
        for a in range(n - 1):
            d = np.minimum(_popcount(h[a] ^ h[a + 1:]), _popcount(h[a] ^ hf[a + 1:]))
            for b in np.nonzero(d <= umbral_hash)[0]:
                pares.append((int(d[b]), a, a + 1 + int(b)))
        pares.sort()
        for _, i, j in pares:
            if uf.find(i) == uf.find(j):
                continue
            if uf.tam_de(i) + uf.tam_de(j) > max_grupo_hash:
                rechazadas += 1
                continue
            unir(i, j, "casi-duplicado visual")

    for i, it in enumerate(items):
        it["grupo"] = uf.find(i)
    causas = {r: uf.causas[r] for r in {it["grupo"] for it in items}}
    return {"razones": razones, "n_grupos": len(causas), "rechazadas_por_tope": rechazadas,
            "causas_por_grupo": causas}


def grupos_mas_grandes(items: list[dict], info: dict, k: int = 5) -> list[dict]:
    """Para el diagnostico: los k grupos mas grandes, con su causa y ejemplos."""
    miembros = defaultdict(list)
    for it in items:
        miembros[it["grupo"]].append(it)
    top = sorted(miembros.items(), key=lambda kv: -len(kv[1]))[:k]
    return [{"tamano": len(m), "originales": len({it["base"] for it in m}),
             "causas": dict(info["causas_por_grupo"].get(g, {})),
             "ejemplos": list(dict.fromkeys(it["base"] for it in m))[:3]} for g, m in top]


def fuga_en_split_original(items: list[dict]) -> dict:
    """Cuantos grupos estan repartidos entre varios splits en el export original."""
    splits_por_grupo = defaultdict(set)
    for it in items:
        splits_por_grupo[it["grupo"]].add(it["split_orig"])
    cruzados = {g for g, s in splits_por_grupo.items() if len(s) > 1}
    imgs_eval_contaminadas = sum(1 for it in items
                                 if it["split_orig"] in ("val", "test") and it["grupo"] in cruzados)
    total_eval = sum(1 for it in items if it["split_orig"] in ("val", "test"))
    return {"grupos_cruzados": len(cruzados), "imgs_eval_contaminadas": imgs_eval_contaminadas,
            "total_eval": total_eval}


# ------------------------------------------------------------ split por grupos
def split_estratificado_por_grupos(items: list[dict], proporciones: dict[str, float],
                                   semilla: int = 42) -> dict[int, str]:
    """Reparte GRUPOS (no imagenes) respetando la proporcion de cada clase.

    Heuristica voraz: grupos grandes primero; cada grupo va al split que mas
    'deficit' relativo tiene para las clases que ese grupo contiene. Cuenta
    ORIGINALES, no archivos: las copias aumentadas no deben inflar un grupo.
    """
    comp = defaultdict(Counter)
    vistos = set()
    for it in items:
        if (it["grupo"], it["base"]) not in vistos:
            vistos.add((it["grupo"], it["base"]))
            comp[it["grupo"]][it["clase"]] += 1
    total = Counter()
    for c in comp.values():
        total.update(c)

    rng = random.Random(semilla)
    orden = list(comp)
    rng.shuffle(orden)
    orden.sort(key=lambda g: -sum(comp[g].values()))   # sort estable: los empates quedan barajados

    cuenta = {s: Counter() for s in proporciones}
    asignado = {}
    for g in orden:
        def deficit(s: str) -> float:
            return sum(n * (proporciones[s] - cuenta[s][c] / max(1, total[c])) for c, n in comp[g].items())
        s = max(proporciones, key=deficit)
        asignado[g] = s
        cuenta[s].update(comp[g])
    return asignado


def revisar_reparto(items: list[dict], asignado: dict[int, str], proporciones: dict[str, float],
                    info: dict, minimo_relativo: float = 0.6) -> dict[str, float]:
    """Se NIEGA a seguir si algun split quedo muy por debajo de su proporcion.

    Esto es lo que habria evitado el primer entrenamiento sobre un val de 227
    imagenes: un reparto desbalanceado no puede pasar en silencio.
    """
    por_split = defaultdict(set)
    for it in items:
        por_split[asignado[it["grupo"]]].add(it["base"])
    total = sum(len(v) for v in por_split.values())
    real = {s: len(por_split.get(s, ())) / max(1, total) for s in proporciones}
    malos = [s for s in proporciones if real[s] < minimo_relativo * proporciones[s]]
    if malos:
        top = grupos_mas_grandes(items, info)
        detalle = "\n".join(f"    {g['originales']:>6} originales  causas={g['causas']}  ej={g['ejemplos']}"
                            for g in top)
        raise RuntimeError(
            "El reparto quedo desbalanceado: "
            + ", ".join(f"{s}={100 * real[s]:.1f}% (objetivo {100 * proporciones[s]:.0f}%)" for s in proporciones)
            + "\n  Grupos mas grandes:\n" + detalle
            + "\n  Si domina 'casi-duplicado visual': baja umbral_hash o max_grupo_hash."
            + "\n  Si domina 'frame del mismo video': hay pocos videos muy largos; no se puede"
            + " separar sin fuga, reduce la proporcion de val/test o consigue mas videos."
        )
    return real


# --------------------------------------------------------------- materializar
def materializar(items: list[dict], asignado: dict[int, str], destino: Path,
                 sobremuestrear_hasta: float = 0.0, una_copia_en_train: bool = False,
                 semilla: int = 42, log=print) -> dict:
    """Copia a <destino>/<split>/<clase>/.

    - val/test: UNA imagen por original (las copias aumentadas no cuentan dos
      veces el mismo huevo), prefiriendo la que venia de valid/test (sin
      aumentacion).
    - train: todas las copias, o una por original si una_copia_en_train=True
      (con entrenamientos largos, las copias duplicadas hacen que cada "epoca"
      sean ~2 pasadas por cada huevo y aceleran la memorizacion).
    - sobremuestreo opcional de clases minoritarias, solo en train.
    """
    if destino.exists():
        shutil.rmtree(destino)
    por_split = defaultdict(list)
    for it in items:
        por_split[asignado[it["grupo"]]].append(it)

    final = {}
    for split, lista in por_split.items():
        if split in ("val", "test") or (split == "train" and una_copia_en_train):
            elegido = {}
            for it in sorted(lista, key=lambda x: (x["split_orig"] == "train", x["ruta"].name)):
                elegido.setdefault(it["base"], it)          # el primero gana: el no aumentado
            lista = list(elegido.values())
        final[split] = lista

    if sobremuestrear_hasta > 0 and "train" in final:
        rng = random.Random(semilla)
        cuenta = Counter(it["clase"] for it in final["train"])
        maximo = max(cuenta.values())
        extra = []
        for clase, n in sorted(cuenta.items()):
            meta = int(maximo * sobremuestrear_hasta)
            if n < meta:
                pool = [it for it in final["train"] if it["clase"] == clase]
                for k in range(meta - n):
                    it = dict(rng.choice(pool))
                    it["dup"] = k + 1
                    extra.append(it)
                log(f"  sobremuestreo train: {clase} {n} -> {meta}")
        final["train"] = final["train"] + extra

    usados = set()
    for split, lista in final.items():
        for it in lista:
            carpeta = destino / split / it["clase"]
            carpeta.mkdir(parents=True, exist_ok=True)
            nombre = it["ruta"].name
            if "dup" in it:
                nombre = f"{Path(nombre).stem}__dup{it['dup']}{Path(nombre).suffix}"
            clave = (split, it["clase"], nombre)
            if clave in usados:
                nombre = f"{it['split_orig']}__{nombre}"
                clave = (split, it["clase"], nombre)
            usados.add(clave)
            shutil.copy2(it["ruta"], carpeta / nombre)
    return final


def verificar_sin_fuga(final: dict, umbral_hash: int = 16, tolerancia_visual: float = 0.02,
                       log=print) -> bool:
    """Prueba independiente de fuga.

    - DURO: ningun original ni grupo puede estar en dos splits.
    - BLANDO: parejas visualmente casi identicas entre val/test y train. Algunas
      son esperables (las que el tope no dejo unir, casi siempre huevos distintos
      con el mismo fondo); solo es fallo si superan `tolerancia_visual` de val+test.
    """
    ok = True
    for campo in ("base", "grupo"):
        dueno = {}
        for split, lista in final.items():
            for it in lista:
                previo = dueno.setdefault(it[campo], split)
                if previo != split:
                    log(f"  FUGA por {campo}: {it[campo]} esta en {previo} y en {split}")
                    ok = False
    tr = final.get("train", [])
    ev = [it for s in ("val", "test") for it in final.get(s, [])]
    if tr and ev and "h" in tr[0]:
        htr = np.stack([it["h"] for it in tr])
        cerca = 0
        for it in ev:
            d = min(int(_popcount(it["h"] ^ htr).min()), int(_popcount(it["hf"] ^ htr).min()))
            cerca += d <= umbral_hash
        frac = cerca / len(ev)
        log(f"  parecidas a algo de train (<= {umbral_hash} bits): {cerca}/{len(ev)} = {100 * frac:.1f}%"
            f"  (tolerancia {100 * tolerancia_visual:.0f}%)")
        if frac > tolerancia_visual:
            log("  FUGA visual: demasiadas imagenes de val/test casi identicas a train")
            ok = False
    return ok


def ic95_proporcion(k: int, n: int) -> tuple[float, float]:
    """Intervalo de Wilson al 95% (funciona bien con proporciones pequenas)."""
    if n == 0:
        return (0.0, 1.0)
    z, p = 1.96, k / n
    den = 1 + z * z / n
    centro = (p + z * z / (2 * n)) / den
    margen = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (max(0.0, centro - margen), min(1.0, centro + margen))

In [ ]:
# @title 4. Descargar el dataset desde Roboflow
from google.colab import userdata
try:
    API_KEY = userdata.get("ROBOFLOW_API_KEY")
except Exception as e:
    raise RuntimeError(
        "Falta la API key. Panel izquierdo > icono de llave (Secretos) > agrega ROBOFLOW_API_KEY "
        "con tu clave de https://app.roboflow.com/settings/api y activa 'Acceso al notebook'."
    ) from e

from roboflow import Roboflow
rf = Roboflow(api_key=API_KEY)
ds = rf.workspace(WORKSPACE).project(PROYECTO).version(VERSION).download(
    "folder", location=str(ORIGEN), overwrite=True)
print("\ndescargado en:", ds.location)
for d in sorted(p for p in ORIGEN.rglob("*") if p.is_dir()):
    n = sum(1 for f in d.iterdir() if f.is_file())
    if n:
        print(f"  {str(d.relative_to(ORIGEN)):<28} {n:>6} archivos")

In [ ]:
# @title 5. Inventario y mapeo de clases al proyecto
import importlib, cdd_split_lib as L
importlib.reload(L)
from collections import Counter

MAPEOS = {
    "3clases":            {"broken": "Roto", "good": "Sano", "dirty": "Sucio"},
    "binario":            {"broken": "Roto", "good": "Sano", "dirty": "Sano"},
    "binario_sin_sucios": {"broken": "Roto", "good": "Sano", "dirty": None},
}
mapeo = MAPEOS[MODO_CLASES]

todos = L.inventariar(ORIGEN)
encontradas = sorted({it["clase_orig"] for it in todos})
sin_mapeo = [c for c in encontradas if c.lower() not in mapeo]
if sin_mapeo:
    raise RuntimeError(f"Clases sin mapeo: {sin_mapeo}. El dataset trae: {encontradas}. Ajusta MAPEOS.")

items = []
for it in todos:
    destino = mapeo[it["clase_orig"].lower()]
    if destino:
        it["clase"] = destino
        items.append(it)

print(f"archivos: {len(todos)}  |  usados en modo '{MODO_CLASES}': {len(items)}")
print(f"originales distintos (sin contar copias aumentadas): {len({it['base'] for it in items})}\n")
tabla = Counter((it["split_orig"], it["clase_orig"]) for it in todos)
print(f"{'clase original':<16}" + "".join(f"{s:>9}" for s in ("train", "val", "test")) + "   -> proyecto")
for c in encontradas:
    print(f"{c:<16}" + "".join(f"{tabla[(s, c)]:>9}" for s in ("train", "val", "test"))
          + f"   -> {mapeo[c.lower()] or '(descartada)'}")

In [ ]:
# @title 6. ¿Hay fuga en el split publicado? (agrupar lo que es el mismo huevo)
# Hash de 256 bits con umbral 16 y tope de 30 por grupo: calibrado sobre 968 fotos reales.
# (La primera version usaba 64 bits y sin tope: encadeno el 91.5% del dataset en un grupo.)
L.calcular_hashes(items, lado=16)
info = L.agrupar(items, gap_rafaga_s=20, duracion_max_rafaga_s=120, umbral_hash=16, max_grupo_hash=30)

print(f"\n{len({it['base'] for it in items})} originales -> {info['n_grupos']} grupos 'mismo huevo'")
print("uniones por causa:")
for k, v in info["razones"].most_common():
    print(f"  {k:<26} {v:>6}")
print(f"  uniones por hash rechazadas por el tope: {info['rechazadas_por_tope']}")

print("\ngrupos mas grandes (si uno solo tiene una fraccion grande del dataset, algo anda mal):")
for g in L.grupos_mas_grandes(items, info):
    print(f"  {g['originales']:>6} originales ({100*g['originales']/len({it['base'] for it in items}):4.1f}%)"
          f"  {g['causas']}  ej: {g['ejemplos']}")

fuga = L.fuga_en_split_original(items)
print("\n=== FUGA EN EL SPLIT PUBLICADO ===")
print(f"grupos repartidos entre varios splits: {fuga['grupos_cruzados']}")
print(f"imagenes de val/test que comparten huevo con train: "
      f"{fuga['imgs_eval_contaminadas']} de {fuga['total_eval']} "
      f"({100*fuga['imgs_eval_contaminadas']/max(1,fuga['total_eval']):.1f}%)")
print("-> cualquier metrica reportada sobre ese test esta inflada en esa proporcion." if fuga["grupos_cruzados"]
      else "-> el split publicado no tenia fuga detectable.")

In [ ]:
# @title 7. Split nuevo 70/15/15 por grupos, estratificado, y verificación independiente
import csv
asignado = L.split_estratificado_por_grupos(items, PROPORCIONES, semilla=SEMILLA)
# Se DETIENE aqui si algun split queda muy por debajo de su proporcion (lo que paso en el
# primer entrenamiento: val=227 y test=226 imagenes). Mejor un error ahora que 1 h de GPU tirada.
reparto = L.revisar_reparto(items, asignado, PROPORCIONES, info)
print("reparto de originales:", {k: f"{100*v:.1f}%" for k, v in reparto.items()})
final = L.materializar(items, asignado, LIMPIO, sobremuestrear_hasta=SOBREMUESTREO,
                       una_copia_en_train=UNA_COPIA_EN_TRAIN, semilla=SEMILLA)

print("\n=== VERIFICACION ===")
sin_fuga = L.verificar_sin_fuga(final, umbral_hash=16)
print("CERO fuga: ningun original, grupo ni casi-duplicado cruza splits" if sin_fuga
      else "ATENCION: revisa los mensajes de fuga de arriba")

clases = sorted({it["clase"] for it in items})
unicos = {s: Counter(it["clase"] for it in {i["base"]: i for i in final[s]}.values()) for s in final}
total = Counter()
for c in unicos.values():
    total.update(c)
print(f"\n{'clase':<8}" + "".join(f"{s:>16}" for s in ("train", "val", "test")))
for c in clases:
    print(f"{c:<8}" + "".join(f"{unicos[s][c]:>8} ({100*unicos[s][c]/total[c]:4.1f}%)" for s in ("train", "val", "test")))
print(f"\narchivos en disco: " + ", ".join(f"{s}={len(final[s])}" for s in ("train", "val", "test"))
      + "  (train incluye copias aumentadas y sobremuestreo)")

n_roto_test = unicos["test"].get("Roto", 0)
lo, hi = L.ic95_proporcion(round(0.03 * n_roto_test), n_roto_test)
print(f"\nCon {n_roto_test} rotos en test, una fuga real de 3% se mediria con IC95 ~[{100*lo:.1f}%, {100*hi:.1f}%]")

# Manifiesto: que archivo fue a que split (reproducibilidad y auditoria)
with open(LIMPIO / "manifiesto_split.csv", "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["split", "clase", "archivo", "original", "grupo", "split_publicado"])
    for s, lista in final.items():
        for it in lista:
            w.writerow([s, it["clase"], it["ruta"].name, it["base"], it["grupo"], it["split_orig"]])
print("manifiesto:", LIMPIO / "manifiesto_split.csv")

In [ ]:
# @title 8. Vista previa: revisa a ojo que las etiquetas tengan sentido
import matplotlib.pyplot as plt, random
from PIL import Image
random.seed(SEMILLA)
fig, ejes = plt.subplots(len(clases), 8, figsize=(16, 2.2 * len(clases)))
for fila, c in enumerate(clases):
    muestras = random.sample(sorted((LIMPIO / "train" / c).glob("*")), 8)
    for col, f in enumerate(muestras):
        ax = ejes[fila][col] if len(clases) > 1 else ejes[col]
        ax.imshow(Image.open(f)); ax.set_xticks([]); ax.set_yticks([])
        if col == 0:
            ax.set_ylabel(c, fontsize=13)
plt.suptitle("Muestras de train por clase", fontsize=14); plt.tight_layout(); plt.show()

## Hiperparámetros: por qué no son los de fábrica

| parámetro | fábrica | aquí | por qué |
|---|---|---|---|
| `scale` | 0.5 | **0.25** | El recorte aleatorio conserva ≥75 % de la imagen (fábrica: ≥50 %; ImageNet puro: 8 %). Un recorte chico puede dejar **fuera la fisura** y mantener la etiqueta "Roto": ruido de etiqueta. |
| `erasing` | 0.4 | **0.15** | Borrar un rectángulo al azar regulariza, pero en un defecto pequeño puede borrar la única evidencia. |
| `flipud` | 0.0 | **0.5** | Un huevo no tiene "arriba": el volteo vertical es aumentación válida y gratis. |
| `hsv_v` | 0.4 | **0.45** | Más variación de brillo: en campo la sobreexposición hizo fallar al detector. |
| `dropout` | 0.0 | **0.15** | Regulariza la cabeza de clasificación con ~3.7k originales. |
| `optimizer` | auto | **AdamW, lr0=5e-4, cos_lr** | *Fine-tuning* estable desde pesos de ImageNet. |
| `auto_augment` | randaugment | randaugment | Incluye variaciones de nitidez, que ayudan contra el desenfoque. |

In [ ]:
# @title 9. Entrenar: mínimo 150 épocas; para si pasan 30 sin mejora significativa
# Si Colab se desconecto a mitad: vuelve a correr las celdas 1 a 8 (recrean el mismo split,
# todo usa SEMILLA), pega aqui el nombre de la carpeta del run viejo (esta en
# MyDrive/eggdetection/runs_cdd/) y corre esta celda y las siguientes.
REANUDAR = ""  # @param {type:"string"}

import csv
from ultralytics import YOLO

def parada_con_minimo(trainer):
    # Ultralytics no trae "minimo de epocas" ni "mejora minima": se implementan aqui.
    # Corre al final de cada epoca, ANTES de que el entrenador decida si sigue.
    # Recalcula todo desde results.csv, asi que funciona igual al reanudar.
    if not Path(trainer.csv).exists():
        return
    with open(trainer.csv, encoding="utf-8") as fh:
        filas = [{k.strip(): v for k, v in f.items()} for f in csv.DictReader(fh)]
    col = next((k for k in filas[0] if "top1" in k), None) if filas else None
    if col is None:
        return
    hist = [float(f[col]) for f in filas]
    ref, ref_ep = hist[0], 1                   # ultima mejora SIGNIFICATIVA
    for ep, v in enumerate(hist, 1):
        if v > ref + MEJORA_MINIMA:
            ref, ref_ep = v, ep
    epoca, sin_mejora = len(hist), len(hist) - ref_ep
    estado = (f"  [parada] epoca {epoca}: top1 val={hist[-1]:.4f} | referencia {ref:.4f} (epoca {ref_ep})"
              f" | {sin_mejora}/{PACIENCIA} sin mejorar >= {MEJORA_MINIMA:g}")
    if epoca < MIN_EPOCAS:
        print(estado + f" | minimo {MIN_EPOCAS}: sigue")
    elif sin_mejora >= PACIENCIA:
        trainer.stop = True
        print(estado + " -> SE DETIENE: paso el minimo y lleva la paciencia completa sin mejora")
    else:
        print(estado)

if REANUDAR:
    RUN = CARPETA_RUNS / REANUDAR
    assert (RUN / "weights" / "last.pt").exists(), f"No encuentro {RUN / 'weights' / 'last.pt'}"
    print("reanudando desde", RUN / "weights" / "last.pt")
    modelo = YOLO(str(RUN / "weights" / "last.pt"))
    modelo.add_callback("on_fit_epoch_end", parada_con_minimo)
    modelo.train(resume=True)
else:
    modelo = YOLO(MODELO_BASE)
    modelo.add_callback("on_fit_epoch_end", parada_con_minimo)
    modelo.train(
        data=str(LIMPIO), imgsz=IMGSZ, batch=BATCH,
        epochs=EPOCAS_MAX,   # tope; el callback decide cuando parar despues de MIN_EPOCAS
        patience=0,          # 0 = apaga la parada de fabrica (paraba a las 15 sin mejora)
        # En el run anterior train/loss llego a 0.035 mientras val caia de 0.868 a 0.75:
        # sobreajuste. Para 150+ epocas hace falta mas regularizacion, no menos.
        dropout=0.3, weight_decay=0.01,
        optimizer="AdamW", lr0=5e-4, cos_lr=True, warmup_epochs=2,
        scale=0.25, erasing=0.15, fliplr=0.5, flipud=0.5, hsv_h=0.015, hsv_s=0.5, hsv_v=0.45,
        auto_augment="randaugment",
        cache="ram", workers=2, seed=SEMILLA, deterministic=False,
        project=str(CARPETA_RUNS), name=NOMBRE_RUN, exist_ok=True, plots=True,
    )
    RUN = Path(modelo.trainer.save_dir)
print("\npesos:", RUN / "weights" / "best.pt")

In [ ]:
# @title 10. Curvas: ¿aprendió o memorizó?
import pandas as pd
df = pd.read_csv(RUN / "results.csv"); df.columns = [c.strip() for c in df.columns]
col_acc = next(c for c in df.columns if "top1" in c)
col_tr  = next(c for c in df.columns if c.startswith("train/") and "loss" in c)
col_va  = next(c for c in df.columns if c.startswith("val/") and "loss" in c)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
a1.plot(df["epoch"], df[col_tr], label="train"); a1.plot(df["epoch"], df[col_va], label="val")
a1.set_title("Pérdida"); a1.set_xlabel("época"); a1.legend()
a2.plot(df["epoch"], df[col_acc]); a2.set_title("Exactitud top-1 en VAL"); a2.set_xlabel("época")
plt.tight_layout(); plt.show()

mejor = int(df[col_acc].idxmax())
ult = df.iloc[-1]
print(f"mejor epoca: {int(df.loc[mejor,'epoch'])}  top1 val = {df.loc[mejor, col_acc]:.4f}")
print(f"ultima epoca: {int(ult['epoch'])}  train/loss={ult[col_tr]:.4f}  val/loss={ult[col_va]:.4f}")
brecha = ult[col_va] - ult[col_tr]
print("-> sin senales fuertes de sobreajuste" if brecha < 0.3 else
      f"-> OJO: val/loss supera a train/loss por {brecha:.2f}: posible sobreajuste")

In [ ]:
# @title 11. Evaluación en TEST con las métricas del proyecto
import numpy as np, json
modelo = YOLO(str(RUN / "weights" / "best.pt"))
NOMBRES = [modelo.names[i] for i in range(len(modelo.names))]
IDX = {n: i for i, n in enumerate(NOMBRES)}
R, S = IDX["Roto"], IDX["Sano"]

def cargar(split):
    rutas, y = [], []
    for i, n in enumerate(NOMBRES):
        for f in sorted((LIMPIO / split / n).glob("*")):
            rutas.append(str(f)); y.append(i)
    return rutas, np.array(y)

def predecir(fuente, lote=64):
    # Lotes a mano: con una lista de imagenes PIL, ultralytics las mete TODAS en un
    # solo lote (LoadPilAndNumpy); con cientos de imagenes eso puede agotar la GPU.
    if not len(fuente):
        return np.zeros((0, len(NOMBRES)), dtype=np.float32)
    out = []
    for i in range(0, len(fuente), lote):
        for r in modelo.predict(fuente[i:i + lote], imgsz=IMGSZ, verbose=False, stream=True):
            out.append(r.probs.data.float().cpu().numpy())
    return np.stack(out)

rutas_val, y_val = cargar("val");   P_val = predecir(rutas_val)
rutas_test, y_test = cargar("test"); P_test = predecir(rutas_test)

# Ambos cortes se calibran en VAL. Elegirlos mirando test seria hacer trampa:
# el numero de test dejaria de ser una estimacion honesta.
p_roto_val = P_val[:, R]
roto_val, bueno_val = (y_val == R), (y_val != R)

# --- 1) TAU_ROTO: a partir de aqui es DANADO. Minimo costo 12:1 en val. ---
mejor = None
for tau in np.round(np.arange(0.05, 0.96, 0.01), 2):
    d = p_roto_val >= tau
    costo = int((roto_val & ~d).sum()) * COSTO_FN + int((bueno_val & d).sum()) * COSTO_FP
    if mejor is None or costo < mejor[0]:          # en empate gana el mas bajo: el mas prudente
        mejor = (costo, float(tau))
TAU_ROTO = mejor[1]

# --- 2) TAU_DUDA: por debajo es BUENO. La banda [TAU_DUDA, TAU_ROTO) es REVISION. ---
# Se abre la banda hacia abajo mientras no mande a revision mas del MAX_REVISION_BUENOS
# de los huevos buenos de val. Mas ancha = menos rotos se cuelan, mas trabajo humano.
TAU_DUDA = TAU_ROTO
for t in np.round(np.arange(TAU_ROTO - 0.01, 0.0, -0.01), 2):
    en_banda = bueno_val & (p_roto_val >= t) & (p_roto_val < TAU_ROTO)
    if en_banda.sum() / max(1, bueno_val.sum()) > MAX_REVISION_BUENOS:
        break
    TAU_DUDA = float(t)

print(f"Calibrado en VAL ({len(y_val)} imagenes):")
print(f"  DANADO     si p(Roto) >= {TAU_ROTO:.2f}   (costo minimo {COSTO_FN:g}:{COSTO_FP:g})")
print(f"  REVISION   si {TAU_DUDA:.2f} <= p(Roto) < {TAU_ROTO:.2f}   (<= {100*MAX_REVISION_BUENOS:.0f}% de los buenos)")
print(f"  BUENO      si p(Roto) < {TAU_DUDA:.2f}" + ("   -> y si p(Sucio) > p(Sano): BUENO (SUCIO)" if "Sucio" in IDX else ""))
print()

def veredicto(p):
    # Jerarquico, como en planta: primero danado o no; solo entre los buenos, limpio o sucio.
    if p[R] >= TAU_ROTO:
        return "DANADO"
    if p[R] >= TAU_DUDA:
        return "REVISION"
    if "Sucio" in IDX and p[IDX["Sucio"]] > p[S]:
        return "BUENO_SUCIO"
    return "BUENO"

def rutear(P):
    return np.array([veredicto(p) for p in P], dtype=object)

rutas_banda = rutear(P_test)
pred = P_test.argmax(1)

# --- metricas clasicas ---
print(f"exactitud top-1 en TEST: {(pred == y_test).mean():.4f}   ({len(y_test)} imagenes)\n")
print(f"{'clase':<8}{'P':>8}{'R':>8}{'F1':>8}{'n':>7}")
for i, n in enumerate(NOMBRES):
    tp = int(((pred == i) & (y_test == i)).sum()); fp = int(((pred == i) & (y_test != i)).sum())
    fn = int(((pred != i) & (y_test == i)).sum())
    p = tp / max(1, tp + fp); r = tp / max(1, tp + fn); f1 = 2 * p * r / max(1e-9, p + r)
    print(f"{n:<8}{p:>8.3f}{r:>8.3f}{f1:>8.3f}{int((y_test == i).sum()):>7}")

# --- matriz de confusion ---
M = np.zeros((len(NOMBRES), len(NOMBRES)), int)
for t, p in zip(y_test, pred):
    M[t, p] += 1
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(M / np.maximum(1, M.sum(1, keepdims=True)), cmap="Blues", vmin=0, vmax=1)
for i in range(len(NOMBRES)):
    for j in range(len(NOMBRES)):
        ax.text(j, i, M[i, j], ha="center", va="center", color="white" if M[i, j] > M[i].sum() / 2 else "black")
ax.set_xticks(range(len(NOMBRES)), NOMBRES); ax.set_yticks(range(len(NOMBRES)), NOMBRES)
ax.set_xlabel("predicho"); ax.set_ylabel("real"); ax.set_title("Matriz de confusión (test)")
plt.tight_layout(); plt.savefig(RUN / "confusion_test.png", dpi=150); plt.show()

# --- lo que de verdad importa: que veredicto recibe cada tipo de huevo ---
VEREDICTOS = ["BUENO", "BUENO_SUCIO", "REVISION", "DANADO"] if "Sucio" in IDX else ["BUENO", "REVISION", "DANADO"]
print(f"\nVEREDICTOS EN TEST (filas = lo que el huevo es en realidad)")
print(f"{'real':<8}" + "".join(f"{v:>14}" for v in VEREDICTOS))
for i, n in enumerate(NOMBRES):
    m = y_test == i
    print(f"{n:<8}" + "".join(f"{int((rutas_banda[m] == v).sum()):>8} {100*(rutas_banda[m] == v).mean():>4.0f}%" for v in VEREDICTOS))

m_roto, m_bueno = (y_test == R), (y_test != R)
n_roto, n_bueno = int(m_roto.sum()), int(m_bueno.sum())
fuga = int((rutas_banda[m_roto] != "DANADO").sum())
a_empaque = int(np.isin(rutas_banda[m_roto], ["BUENO", "BUENO_SUCIO"]).sum())
buenos_a_revision = int((rutas_banda[m_bueno] == "REVISION").sum())
buenos_descartados = int((rutas_banda[m_bueno] == "DANADO").sum())
lo, hi = L.ic95_proporcion(fuga, n_roto)
lo2, hi2 = L.ic95_proporcion(a_empaque, n_roto)
print(f"\nrotos que NO salen como DANADO:   {fuga}/{n_roto} = {100*fuga/max(1,n_roto):.2f}%  IC95 [{100*lo:.1f}%, {100*hi:.1f}%]")
print(f"  de esos, llegan al EMPAQUE (BUENO): {a_empaque}/{n_roto} = {100*a_empaque/max(1,n_roto):.2f}%  IC95 [{100*lo2:.1f}%, {100*hi2:.1f}%]")
print(f"  el resto cae en REVISION, que es justo para lo que existe")
print(f"buenos mandados a REVISION:       {buenos_a_revision}/{n_bueno} = {100*buenos_a_revision/max(1,n_bueno):.1f}%")
print(f"buenos botados como DANADO:       {buenos_descartados}/{n_bueno} = {100*buenos_descartados/max(1,n_bueno):.1f}%")
if "Sucio" in IDX:
    ok_sub = sum(1 for t, v in zip(y_test, rutas_banda)
                 if v in ("BUENO", "BUENO_SUCIO") and t != R and (v == "BUENO_SUCIO") == (t == IDX["Sucio"]))
    tot_sub = int(np.isin(rutas_banda[m_bueno], ["BUENO", "BUENO_SUCIO"]).sum())
    print(f"limpio vs sucio, entre los buenos: {ok_sub}/{tot_sub} bien distinguidos")

METRICAS = {"top1_test": float((pred == y_test).mean()), "tau_roto": TAU_ROTO, "tau_duda": TAU_DUDA,
            "fuga_pct": 100 * fuga / max(1, n_roto), "fuga_ic95": [100 * lo, 100 * hi],
            "fuga_a_empaque_pct": 100 * a_empaque / max(1, n_roto),
            "fuga_a_empaque_ic95": [100 * lo2, 100 * hi2],
            "buenos_a_revision_pct": 100 * buenos_a_revision / max(1, n_bueno),
            "buenos_descartados_pct": 100 * buenos_descartados / max(1, n_bueno),
            "n_test": int(len(y_test)), "n_roto_test": n_roto, "clases": NOMBRES, "modo": MODO_CLASES}

In [ ]:
# @title 12. Robustez: las MISMAS pruebas que tumbaron al detector en campo
from PIL import ImageDraw, ImageFilter, ImageEnhance
import random

def rayar(im, semilla=3):
    # linea oscura irregular por el centro, imitando un lapicero sobre la cascara
    im = im.copy(); d = ImageDraw.Draw(im); w, h = im.size; rng = random.Random(semilla)
    pts = [(w / 2 + w * 0.12 * np.sin(t * 3.1) + rng.uniform(-2, 2), h * (0.2 + 0.6 * t))
           for t in (i / 13 for i in range(14))]
    d.line(pts, fill=(35, 25, 20), width=max(2, round(w / 110)), joint="curve")
    return im

# --- raya de lapicero sobre huevos SANOS bien clasificados ---
# Con la clase Sucio hay un desenlace nuevo e interesante: que el modelo diga
# "BUENO (SUCIO)", o sea "es una marca, no un dano". Eso seria el acierto ideal.
sanos_ok = [rutas_test[i] for i in range(len(y_test)) if y_test[i] == S and pred[i] == S]
limpios = [Image.open(f).convert("RGB") for f in sanos_ok]
P0 = predecir(limpios); P1 = predecir([rayar(im) for im in limpios])
v_raya = rutear(P1)
volteados = int((v_raya == "DANADO").sum())
print("=== RAYA DE LAPICERO sobre huevos sanos ===")
for v in VEREDICTOS:
    print(f"  -> {v:<12} {int((v_raya == v).sum()):>4} / {len(limpios)}")
print(f"  botados como DANADO solo por la raya: {100*volteados/max(1,len(limpios)):.1f}%")
print(f"  subida media de p(Roto): {np.mean(P1[:, R] - P0[:, R]):+.3f}")
print(f"  (el detector: 6/20 = 30% a descarte, subida media +0.261)\n")

# --- desenfoque sobre huevos ROTOS (radio escalado: equivalente al de una imagen de 640 px) ---
rotos = [Image.open(rutas_test[i]).convert("RGB") for i in range(len(y_test)) if y_test[i] == R]
print("=== DESENFOQUE sobre huevos rotos ===")
print(f"{'radio equiv. 640px':>19}{'p(Roto)':>9}{'DANADO':>9}{'REVISION':>10}{'BUENO*':>8}{'no DANADO':>11}")
ROBUSTEZ = {"raya_pct": 100 * volteados / max(1, len(limpios)),
            "raya_veredictos": {v: int((v_raya == v).sum()) for v in VEREDICTOS},
            "blur": {}, "brillo": {}}
for r640 in [0, 1, 2, 3, 4, 6]:
    ims = [im.filter(ImageFilter.GaussianBlur(r640 * im.width / 640)) if r640 else im for im in rotos]
    P = predecir(ims); rr = rutear(P)
    no_danado = 100 * (rr != "DANADO").mean()
    ROBUSTEZ["blur"][r640] = no_danado
    bueno = int(np.isin(rr, ["BUENO", "BUENO_SUCIO"]).sum())
    print(f"{r640:>19}{P[:, R].mean():>9.3f}{int((rr == 'DANADO').sum()):>9}{int((rr == 'REVISION').sum()):>10}{bueno:>8}{no_danado:>10.1f}%")
print("  BUENO* = BUENO o BUENO (SUCIO): esos si llegarian al empaque")
print("  (el detector, misma definicion: radio 3 -> 12/25 rotos NO descartados = 48%)\n")

# --- brillo ---
print("=== EXPOSICION sobre huevos rotos ===")
print(f"{'brillo':>8}{'p(Roto)':>9}{'no DANADO':>11}")
for f in [0.35, 0.5, 0.7, 1.0, 1.4, 1.8, 2.4]:
    P = predecir([ImageEnhance.Brightness(im).enhance(f) for im in rotos]); rr = rutear(P)
    ROBUSTEZ["brillo"][f] = 100 * (rr != "DANADO").mean()
    print(f"{f:>8.2f}{P[:, R].mean():>9.3f}{ROBUSTEZ['brillo'][f]:>10.1f}%")
print("  (el detector, misma definicion: brillo 2.4x -> 15/25 rotos NO descartados = 60%)")

In [ ]:
# @title 13. Exportar a ONNX y verificar que da lo mismo que PyTorch
import onnxruntime as ort
onnx_path = modelo.export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True, dynamic=False)
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
print("entrada:", sess.get_inputs()[0].name, sess.get_inputs()[0].shape,
      "| salida:", sess.get_outputs()[0].name, sess.get_outputs()[0].shape)

def preproc(ruta):
    # Replica EXACTA de classify_transforms (ultralytics 8.4.157) para que el backend haga lo mismo:
    # 1) lado corto -> 224 (bilineal)  2) recorte central 224x224  3) /255, RGB, CHW. Sin normalizar media.
    im = Image.open(ruta).convert("RGB"); w, h = im.size
    if w <= h: nw, nh = IMGSZ, int(IMGSZ * h / w)
    else:      nw, nh = int(IMGSZ * w / h), IMGSZ
    im = im.resize((nw, nh), Image.BILINEAR)
    izq, arr = int(round((nw - IMGSZ) / 2)), int(round((nh - IMGSZ) / 2))
    im = im.crop((izq, arr, izq + IMGSZ, arr + IMGSZ))
    return (np.asarray(im, dtype=np.float32) / 255.0).transpose(2, 0, 1)[None]

n = min(200, len(rutas_test))
P_onnx = np.concatenate([sess.run(None, {sess.get_inputs()[0].name: preproc(f)})[0] for f in rutas_test[:n]])
dif = np.abs(P_onnx - P_test[:n]).max()
coinc = (P_onnx.argmax(1) == P_test[:n].argmax(1)).mean()
print(f"\nONNX vs PyTorch en {n} imagenes: coincidencia top-1 {100*coinc:.1f}%   dif. maxima {dif:.2e}")
assert coinc >= 0.99, "El ONNX no reproduce al modelo: revisar el preprocesamiento"
print("OK: el backend puede usar este ONNX con esa receta de preprocesamiento")

In [ ]:
# @title 14. Guardar todo (Drive o descarga)
import shutil, json
PAQ = Path("/content") / RUN.name       # RUN.name: si se reanudo, es el nombre del run original
if PAQ.exists(): shutil.rmtree(PAQ)
(PAQ / "weights").mkdir(parents=True)
shutil.copy2(RUN / "weights" / "best.pt", PAQ / "weights" / "best.pt")
shutil.copy2(onnx_path, PAQ / "weights" / "best.onnx")
for f in ["results.csv", "results.png", "args.yaml", "confusion_test.png",
          "confusion_matrix.png", "confusion_matrix_normalized.png"]:
    if (RUN / f).exists(): shutil.copy2(RUN / f, PAQ / f)
shutil.copy2(LIMPIO / "manifiesto_split.csv", PAQ / "manifiesto_split.csv")
receta = {"preprocesamiento": "RGB; lado corto a %d bilineal; recorte central %dx%d; /255; CHW; sin normalizar" % (IMGSZ, IMGSZ, IMGSZ),
          "salida": "probabilidades softmax en el orden de 'clases'", "ultralytics": "8.4.157"}
(PAQ / "metricas.json").write_text(json.dumps({**METRICAS, "robustez": ROBUSTEZ, "receta_onnx": receta}, indent=2))
zip_path = shutil.make_archive(str(PAQ), "zip", PAQ)
print("paquete:", zip_path, f"({Path(zip_path).stat().st_size/1e6:.1f} MB)")

if GUARDAR_EN_DRIVE:
    # Drive ya quedo montado en la celda 2 (y el run ya vive ahi); solo falta el zip.
    dest = Path("/content/drive/MyDrive/eggdetection"); dest.mkdir(parents=True, exist_ok=True)
    shutil.copy2(zip_path, dest)
    print("copiado a Drive:", dest / Path(zip_path).name)
else:
    from google.colab import files
    files.download(zip_path)

In [ ]:
# @title 15. Resumen para el informe
print("=" * 70)
print(f"MODELO  {RUN.name}")
print("=" * 70)
print(f"dataset       CDD_eggs_SJ v{VERSION} (CC BY 4.0), split 70/15/15 por grupos, sin fuga")
print(f"clases        {NOMBRES}  (modo {MODO_CLASES})")
print(f"exactitud     {100*METRICAS['top1_test']:.2f}% top-1 en test ({METRICAS['n_test']} imgs)")
print(f"cortes        DANADO si p(Roto) >= {TAU_ROTO:.2f} | REVISION entre {TAU_DUDA:.2f} y {TAU_ROTO:.2f} | BUENO por debajo")
print(f"rotos a EMPAQUE   {METRICAS['fuga_a_empaque_pct']:.2f}%  IC95 [{METRICAS['fuga_a_empaque_ic95'][0]:.1f}, {METRICAS['fuga_a_empaque_ic95'][1]:.1f}]   <- el KPI del proyecto")
print(f"rotos no DANADO   {METRICAS['fuga_pct']:.2f}%  (el resto va a REVISION)")
print(f"buenos a REVISION {METRICAS['buenos_a_revision_pct']:.1f}%   buenos botados {METRICAS['buenos_descartados_pct']:.1f}%")
print()
print(f"{'prueba (menor es mejor)':<38}{'detector YOLOv8m':>18}{'este modelo':>14}")
print(f"{'sanos con raya -> DANADO':<38}{'30.0%':>18}{ROBUSTEZ['raya_pct']:>13.1f}%")
print(f"{'rotos NO DANADO, desenfoque r3':<38}{'48.0%':>18}{ROBUSTEZ['blur'][3]:>13.1f}%")
print(f"{'rotos NO DANADO, brillo 2.4x':<38}{'60.0%':>18}{ROBUSTEZ['brillo'][2.4]:>13.1f}%")
print("=" * 70)
print("Detector: medido sobre el test del dataset 'huevos' (20 sanos / 25 rotos, ver docs/ANALISIS_CAMPO.md).")
print("Son datasets distintos: la comparacion es indicativa, no un A/B controlado.")

In [ ]:
# @title 16. Prueba con TUS fotos de huevos (desde el celular o el PC)
# Encuadra el huevo AL CENTRO y que ocupe buena parte de la foto: el modelo mira solo
# el cuadrado central de la imagen (en produccion ese recorte lo hace el detector).
# Puedes volver a correr esta celda cuantas veces quieras.
import io
from PIL import ImageOps
from google.colab import files

ETIQUETA = {"BUENO": "✓ BUENO", "BUENO_SUCIO": "✓ BUENO (SUCIO)", "REVISION": "⚠ REVISIÓN", "DANADO": "✗ DAÑADO"}
# El color marca la decision de fondo (pasa / duda / se bota); el borde punteado marca
# "sucio" dentro de los buenos. La etiqueta de texto siempre dice el veredicto completo.
COLOR = {"BUENO": "#0ca30c", "BUENO_SUCIO": "#0ca30c", "REVISION": "#fab219", "DANADO": "#d03b3b"}
TRAZO = {"BUENO_SUCIO": (0, (4, 3))}

subidas = files.upload()   # boton "Elegir archivos": puedes seleccionar varias fotos a la vez
if not subidas:
    print("No subiste fotos. Vuelve a correr esta celda cuando quieras probar.")
else:
    # exif_transpose: las fotos del celular vienen giradas por metadatos
    imgs = [ImageOps.exif_transpose(Image.open(io.BytesIO(b))).convert("RGB") for b in subidas.values()]
    vistas = []
    for im in imgs:                       # lo que realmente ve el modelo
        w, h = im.size; lado = min(w, h); x0, y0 = (w - lado) // 2, (h - lado) // 2
        vistas.append(im.crop((x0, y0, x0 + lado, y0 + lado)))
    P = predecir(imgs)

    cols = min(4, len(imgs)); filas = -(-len(imgs) // cols)
    fig, ejes = plt.subplots(filas, cols, figsize=(4 * cols, 4.9 * filas), squeeze=False)
    for ax in ejes.flat:
        ax.set_visible(False)
    for ax, nombre, vista, p in zip(ejes.flat, subidas, vistas, P):
        v = veredicto(p)
        ax.set_visible(True); ax.imshow(vista); ax.set_xticks([]); ax.set_yticks([])
        for borde in ax.spines.values():
            borde.set_edgecolor(COLOR[v]); borde.set_linewidth(7); borde.set_linestyle(TRAZO.get(v, "-"))
        probs = "   ".join(f"{NOMBRES[i]} {p[i]:.0%}" for i in range(len(NOMBRES)))
        ax.set_title(f"{ETIQUETA[v]}\n{probs}", fontsize=10.5, color="#0b0b0b")
        ax.set_xlabel(nombre[:36], fontsize=8, color="#52514e")
    plt.tight_layout(); plt.show()

    print(f"cortes: DANADO si p(Roto) >= {TAU_ROTO:.2f} | REVISION si >= {TAU_DUDA:.2f}\n")
    for nombre, p in zip(subidas, P):
        print(f"{ETIQUETA[veredicto(p)]:<18} p(Roto)={p[R]:.3f}   {nombre}")

## Cómo encaja en el proyecto

Este modelo **no reemplaza** al detector: lo complementa. La arquitectura recomendada es de **dos etapas**:

```
frame → detector YOLOv8m (¿dónde hay huevos?) → recorte CUADRADO con margen
      → clasificador YOLOv8m-cls (¿cómo está?) → política de ruteo → banda
```

- El recorte debe ser **cuadrado**: el preprocesamiento hace recorte central, y en un recorte
  rectangular cortaría parte del huevo.
- Dos modelos entrenados con **datasets independientes** cometen errores menos correlacionados.
  Cuando discrepan (el detector dice Sano y el clasificador Roto, o al revés), eso es una señal
  natural para mandar a `REVISION`.
- El clasificador a 224 px cuesta ~5 GFLOPs contra ~79 del detector a 640: añade poca latencia.

## Opcional: la versión más limpia posible (fork)

La v2 publicada viene **estirada a 224×224** (deforma el óvalo) y con **shear ±10° horneado**.
Para entrenar sobre los originales:

1. En la página del dataset → **Fork Dataset** (gratis, queda en tu workspace).
2. En tu copia → **Generate New Version**:
   - Preprocesamiento: *Auto-Orient* y *Resize → Fit (black edges) in 320×320* (sin estirar).
   - Aumentaciones: **ninguna** (Ultralytics aumenta en línea, distinto en cada época, y val/test quedan limpios).
3. Cambia `WORKSPACE`, `PROYECTO` y `VERSION` en la celda 2 por los de tu copia, y `IMGSZ = 320`.

El split por grupos de este notebook se sigue aplicando igual.

## Cita (obligatoria por la licencia CC BY 4.0)

```
@misc{cdd_eggs_sj_dataset,
  title = {CDD_eggs_SJ Dataset}, type = {Open Source Dataset}, author = {SJ},
  howpublished = {\url{https://universe.roboflow.com/sj-ytwh0/cdd_eggs_sj}},
  journal = {Roboflow Universe}, publisher = {Roboflow}, year = {2026}, month = {sep},
}
```